In [1]:
import warnings
import importlib
import sys

import pandas as pd
import torch
from transformers import AutoTokenizer, BigBirdConfig, BigBirdForMaskedLM
import tqdm

from CodonTransformer.CodonJupyter import (
    UserContainer,
    display_protein_input,
    format_model_output,
)
from CodonTransformer.CodonPrediction import predict_dna_sequence

warnings.filterwarnings("ignore")
# Reload modules to get latest changes
if 'CodonTransformer.CodonJupyter' in sys.modules:
    importlib.reload(sys.modules['CodonTransformer.CodonJupyter'])
    from CodonTransformer.CodonJupyter import format_model_output

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import ipywidgets as widgets
from IPython.display import display

model_picker = widgets.Dropdown(
    options=[("Rice mitochondria model", "mitochondria"), ("Rice nucleic model", "nucleic")],
    value="mitochondria",
    description="Model:",
)
display(model_picker)

Dropdown(description='Model:', options=(('Rice mitochondria model', 'mitochondria'), ('Rice nucleic model', 'n…

In [ ]:
# Load model and tokenizer for the model selected above
tokenizer = AutoTokenizer.from_pretrained("adibvafa/CodonTransformer")

if model_picker.value == "nucleic":
    model = BigBirdForMaskedLM.from_pretrained("gupta-reyansh123/Rice_Nucleic_Optimizer")
else:
    model = BigBirdForMaskedLM.from_pretrained("gupta-reyansh123/Rice_Mitochondria_Optimizer")

**Optimizing a Single Sequence**
-------------------------------------
1. Run the next code cell and input your DNA sequence and select an organism

2. Run the code cell after it to optimize the sequence and display it.

DNA sequences should only contain valid nucleotides (A, T, G, C) and will be automatically converted to a protein sequence for optimization.

In [6]:
user = UserContainer()
user.organism = 0  # Set to organism 0
display_protein_input(user)

HTML(value='\n        <style>\n            .widget-textarea > textarea {\n                font-size: 12px;\n  …

In [7]:
output = predict_dna_sequence(
    protein=user.protein,
    organism=user.organism,
    device=DEVICE,
    tokenizer=tokenizer,
    model=model,
    attention_type="original_full",
    deterministic=True,
)

print(format_model_output(output))

-----------------------------
|       Predicted DNA       |
-----------------------------
ATGGCACTTCTTTGGGCAAGACCATAG


**Optimizing Multiple Sequences**
-------------------------------------
1. Create a CSV file that has columns 'protein_sequence' and 'organism'.
   You can have other columns in any order.

2. Replace the _dataset_path_ below with the actual path to your CSV file.

3. Run the next code cells to optimize and save the predicted DNA sequences.

In [ ]:
# Update with the actual path to your dataset
dataset_path = "ModelTest/nifH_data.csv"
output_path = "ModelTest/model_predictions.csv"

dataset = pd.read_csv(dataset_path, index_col=0)
dataset["predicted_dna"] = None
dataset.head()

In [ ]:
for index, data in tqdm.tqdm(
    dataset.iterrows(),
    desc=f"CodonTransformer Predicting",
    unit=" Sequences",
    total=dataset.shape[0],
):

    outputs = predict_dna_sequence(
        protein=data["protein"],
        organism=0,
        device=DEVICE, 
        tokenizer=tokenizer,
        model=model,
    )
    dataset.loc[index, "predicted_dna"] = outputs.predicted_dna

dataset.to_csv(output_path)
dataset.head()